In [1]:
# importing libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
from lightgbm import LGBMClassifier

train_df = pd.read_pickle("../data/train.pkl")

In [2]:
from sklearn.metrics import (
    accuracy_score ,
    precision_score ,
    recall_score ,
    f1_score ,
    average_precision_score , 
    roc_auc_score ,
    confusion_matrix ,
    classification_report, 
    ConfusionMatrixDisplay 
)

In [ ]:
train_df = train_df.sort_values(["Transaction_Day" , "Transaction_Time_Of_Day"]).reset_index(drop=True)

In [ ]:
# Initializing PIPELINE

In [ ]:
# NUMERICAL

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median' , add_indicator=True)) ,
    ('scaler' , StandardScaler())
])

In [ ]:
# CATEGORICAL

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant' , fill_value="Missing")) ,
    ('encoder' , OneHotEncoder(handle_unknown='ignore'))
])

In [ ]:
x = train_df.drop(columns=["isFraud" , "TransactionID" ,"TransactionDT"])
y = train_df["isFraud"]

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5 , gap=0)

In [ ]:
oof_predictions = np.full(len(x) , np.nan)

for fold , (train_idx , val_idx) in enumerate(tscv.split(x) , start=1):
    x_train = x.iloc[train_idx]
    x_val = x.iloc[val_idx]

    y_train = y.iloc[train_idx].copy()
    y_val = y.iloc[val_idx].copy()

    numeric_cols = x_train.select_dtypes(include=["int64" , "float64" , "uint16" , "uint8"]).columns
    categorical_cols = x_train.select_dtypes(include=["object"]).columns

    # Combined Pipeline
    preprocessor = ColumnTransformer([
        ('num' , numeric_pipeline , numeric_cols) ,
        ('cat' , categorical_pipeline , categorical_cols)
    ])

    lgbmc_pipeline = Pipeline([
        ('preprocessor' , preprocessor) ,
        ('model' , LGBMClassifier( objective="binary" , random_state=42 , n_jobs=-1 , max_depth=-1 , n_estimators=100 , learning_rate = 0.05 ,subsample=0.8 ,min_child_weight = 5, 
                num_leaves=31 , colsample_bytree = 0.8, reg_alpha = 0.0, reg_lambda = 0.0 , min_child_samples=20))
    ])

    lgbmc_pipeline.fit(x_train , y_train)
    val_pred = lgbmc_pipeline.predict(x_val)
    val_prob = lgbmc_pipeline.predict_proba(x_val)[:,1]
    oof_predictions[val_idx] = val_prob

    roc_auc = roc_auc_score(y_val , val_prob)
    pr_auc = average_precision_score(y_val , val_prob)
    recall = recall_score(y_val , val_pred)
    f1 = f1_score(y_val , val_pred)
    precision = precision_score(y_val , val_pred)


    print(
        f"Fold {fold} | "
        f"ROC-AUC : {roc_auc:.4f} | " 
        f"PR-AUC : {pr_auc:.4f} | "
        f"Recall = {recall:.4f} | "
        f"F1 Score = {f1:.4f} | "
        f"Precision = {precision:.4f}")
    print(confusion_matrix(y_val , val_pred))

oof_mask = ~np.isnan(oof_predictions)
overall_roc_auc = roc_auc_score(y.iloc[oof_mask] , oof_predictions[oof_mask])
overall_pr_auc = average_precision_score(y.iloc[oof_mask] , oof_predictions[oof_mask])
print(f"OOF ROC-AUC : {overall_roc_auc:.4f}")
print(f"OOF PR-AUC : {overall_pr_auc:.4f}")

[LightGBM] [Info] Number of positive: 2529, number of negative: 95896
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 3.381190 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 43876
[LightGBM] [Info] Number of data points in the train set: 98425, number of used features: 5248
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.025695 -> initscore=-3.635440
[LightGBM] [Info] Start training from score -3.635440
Fold 1 | ROC-AUC : 0.8908 | PR-AUC : 0.5497 | Recall = 0.3027 | F1 Score = 0.4529 | Precision = 0.8994
[[95004   112]
 [ 2306  1001]]
[LightGBM] [Info] Number of positive: 5836, number of negative: 191012
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 11.275833 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 52049
[LightGBM] [Info] Number of data poin

In [ ]:
# Fold 1 | ROC-AUC : 0.8908 | PR-AUC : 0.5497 | Recall = 0.3027 | F1 Score = 0.4529 | Precision = 0.8994
# [[95004   112]
#  [ 2306  1001]]

# Fold 2 | ROC-AUC : 0.8975 | PR-AUC : 0.5751 | Recall = 0.3699 | F1 Score = 0.5162 | Precision = 0.8538
# [[94108   257]
#  [ 2557  1501]]

# Fold 3 | ROC-AUC : 0.8912 | PR-AUC : 0.5303 | Recall = 0.3104 | F1 Score = 0.4587 | Precision = 0.8786
# [[94579   158]
#  [ 2542  1144]]

# Fold 4 | ROC-AUC : 0.8990 | PR-AUC : 0.5556 | Recall = 0.3467 | F1 Score = 0.4892 | Precision = 0.8304
# [[94476   261]
#  [ 2408  1278]]

# Fold 5 | ROC-AUC : 0.9015 | PR-AUC : 0.5336 | Recall = 0.3371 | F1 Score = 0.4776 | Precision = 0.8190
# [[94773   253]
#  [ 2252  1145]]

# OOF ROC-AUC : 0.8944
# OOF PR-AUC : 0.5470


In [ ]:
lightgbm_oof = pd.DataFrame({
    "isFraud" : y.values ,
    "lightgbm_oof" : oof_predictions
})

lightgbm_oof.to_csv("../outputs/oof/lightgbm_oof.csv" , index=False)